# Quickstart - Data Processing
<span style="font-family: 'Helvetica'; font-size: 15px;">
    Loading and processing ISAC data for pipeline. Code also found in scripts/build_dataset.py
</span>

In [2]:
import sys
import polars as pl
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))



<span style="font-family: 'Helvetica'; font-size: 15px;">
    Raw ISAC simulation data <span style="color:red">(more details from Steve if needed)</span> may contain artifacts or issues that need to be cleaned first. For example, we won't concern ourselves with false positives, so these should be removed from the dataset. Ideally, there should be a target (ISAC simulation inputs), detection (labeled detections from ISAC simulator, can be omitted if only inference is needed), snr (details about mpc), and error dataset. These datasets should be named placed in the data/raw folder.
</span>

In [3]:
from scripts.build_dataset import load_raw

[target, detect, error, snr] = load_raw(path = "data/raw", names = ["target", "detection", "error", "snr"])
print(detect.columns)

['scenario_id', 'drop_id', 'detection_id', 'associated_target_id', 'is_true_positive', 'is_false_positive', 'estimated_x_m', 'estimated_y_m', 'estimated_z_m', 'estimated_delay_s', 'estimated_bistatic_range_m', 'estimated_bistatic_doppler_hz', 'estimated_rx_azimuth_deg', 'estimated_rx_elevation_deg', 'estimated_tx_azimuth_deg', 'estimated_tx_elevation_deg', 'vel_parallel', 'peak_rda']


<span style="font-family: 'Helvetica'; font-size: 15px;">
    Sometimes, the ISAC raw data will not have all the features we need for our models. In this case, we'll have to build our own features, which roughly fall under the categories of nearest neighbors, snr, geometry, and angle features. These can be vary base off of what test is being run, and any additional features, or any that need to be removed can be edited in their respective function
</span>

In [4]:
from scripts.build_dataset import label_error
from src.feature_building.nn import build_nn_features
from src.feature_building.snr import build_snr_features
from src.feature_building.geo import build_geometry_features
from src.feature_building.quantization import build_angle_features

KEYS = ["scenario_id", "drop_id", "target_id"]

error = label_error(error)
snr_features = build_snr_features(snr)
nn_features = build_nn_features(target)

# Some angle and geometry features for testing depend on detection, and also 
# break when false positives are present
detect = detect.rename({"associated_target_id" : "target_id"})
detect = detect.drop_nans("target_id").with_columns(
    pl.col("target_id").cast(pl.Int64)
)
combined = target.join(detect, on = KEYS, how = "left")
angle_features = build_angle_features(combined)
geo_features = build_geometry_features(combined)



<span style="font-family: 'Helvetica'; font-size: 15px;">
    Rebuilding the features ma take long, so its easier to merge all data into a single csv to cache. Our training data contains 50,000 instances across n = 1,2,3,4,5 targets per drop, evenly split across each scenario.
</span>

In [5]:
path = "data/feature_cache/all_labels_features.csv"

merged_raw = (target
                .join(detect, on = KEYS, how = "left")
                .join(error, on = KEYS, how = "left")
                .join(snr_features, on = KEYS, how = "left"))

merged = pl.concat([merged_raw, geo_features, angle_features, nn_features], how = "horizontal")
merged.write_csv(path)

data = pl.read_csv(path)
print(data)

shape: (49_999, 124)
┌────────────┬─────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ scenario_i ┆ drop_id ┆ target_id ┆ target_x_ ┆ … ┆ nn_el_sep ┆ nn_norm_r ┆ nn_norm_a ┆ n_targets │
│ d          ┆ ---     ┆ ---       ┆ m         ┆   ┆ ---       ┆ ange_sep  ┆ ng_sep    ┆ ---       │
│ ---        ┆ i64     ┆ i64       ┆ ---       ┆   ┆ str       ┆ ---       ┆ ---       ┆ i64       │
│ str        ┆         ┆           ┆ f64       ┆   ┆           ┆ str       ┆ str       ┆           │
╞════════════╪═════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ uma_trp1_h ┆ 1       ┆ 1         ┆ 284.80608 ┆ … ┆ null      ┆ null      ┆ null      ┆ 1         │
│ 38901_isac ┆         ┆           ┆ 8         ┆   ┆           ┆           ┆           ┆           │
│ _1_target  ┆         ┆           ┆           ┆   ┆           ┆           ┆           ┆           │
│ uma_trp1_h ┆ 2       ┆ 2         ┆ 87.117879 ┆ … ┆ null      ┆ null 

/tmp/ipykernel_14032/2347763812.py:8: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  merged = pl.concat([merged_raw, geo_features, angle_features, nn_features], how = "horizontal")


<span style="font-family: 'Helvetica'; font-size: 15px;">
    It may also be helpful to preload certain situations that will be called a lot. For example, we might want to pre load the data for positional error prediction. After caching, simply call load_pos(), which will unpack into tags (indices, scenario, drop), labels (angle error, 3d pos error, range error, etc) and feaures. To load independent, pairs, etc, simply call their respective loading function from src/data/load.
</span>

In [13]:
from src.feature_building.feature_selector import process_pos_features
from src.data.load import load_pos

pos = process_pos_features(data)
pos.write_csv("data/feature_cache/pos_features.csv")
tags, labels, features = load_pos(path = "data/feature_cache/", name = "pos_features.csv")


<span style="font-family: 'Helvetica'; font-size: 15px;">
    Finally, the classification problem uses two stage modeling. So we divide the dataset into mutual pairs (those who share identical values across MATCH_COLS) and independent targets. process_indep_feature and process_pair_feature uses match_cols as a parameter and then splits the entire dataset passed, you you can either pass in mutual_pairs, or the full dataset.
</span>

In [ ]:
from src.feature_building.feature_selector import indep_mutual_split, process_indep_features, process_pair_features, process_classifier_features
from polars.testing import assert_frame_equal

MATCH_COLS = [
    "scenario_id",
    "drop_id",
    #"nearest_neighbor_xyz_distance_m",
    #"nearest_neighbor_bistatic_range_sep_m",
    #"nearest_neighbor_rx_az_sep_deg",
    #"nearest_neighbor_rx_el_sep_deg",
    #"nearest_neighbor_tx_az_sep_deg",
    #"nearest_neighbor_tx_el_sep_deg",
    "nearest_neighbor_norm_range_sep",
    "nearest_neighbor_norm_rx_angle_sep",
]

mutual_pairs, non_mutual = indep_mutual_split(data, MATCH_COLS)
indep = process_indep_features(non_mutual, MATCH_COLS)
indep_2 = process_indep_features(data, MATCH_COLS)

assert_frame_equal(indep, indep_2)

mp = process_pair_features(mutual_pairs, MATCH_COLS)
mp_2 = process_pair_features(data, MATCH_COLS)

assert_frame_equal(mp, mp_2)


<span style="font-family: 'Helvetica'; font-size: 15px;">
    There are many requirements each dataframe must have. Each feature must be implemented in a file in feature_building and built, and features should be added to the proper enum in src/data/features.py. There are also many features that are built in our build_angle_features, build_snr_features ... that aren't needed for inference or training, but illustrative for testing. If these aren't needed, feel free to comment any out in their respective file, or manually rebuild. Only features listed under features.py are needed for training. <span style="color:red">create additional building just for training/inference w/o extra testing features</span>
</span>